# H3 Character Swap (Character & Multi-Person Video Replacement Web Studio)

> Bilingual Colab Launcher (**English-primary** with Chinese quick notes · 主英文说明，关键参数附中文注释)

- **Recommended GPUs / 显存要求**: `A100 80GB` / `H100 80GB` / `RTX 6000 96GB` for `736p` & full-speed inference; `A100 40GB` is also supported at `480p` (`832×480`, recommended `5s` clips first).
- **Powered by `MiniMax-H3 Ref2VA INT8 + FL2V Turbo 8-step + Character Swap V1.5 + SAM 3.1`**: Replaces a character or multiple people directly from a source performance video (`<Video 1>`) and standalone character reference photos (`<Picture 1..N>`) via stock `MiniMaxH3ReferenceToVideo` + automatic `SAM 3.1` subject tracking & edit mask + 8-step `er_sde` sampling — locking background pixels and redrawing full head/outfit identity while preserving motion and soundtrack.
- **关键能力说明（人物替换与多人物替换）**：同一页面支持 **人物替换** 与 **多人物替换**（支持 1–4 位角色），内置 `SAM 3.1` 自动人物分割跟踪与 64px 遮罩外扩（无需手绘遮罩，背景完全锁定，完整替换人物脸型与全身服饰），支持可选关键帧锚点引导（`Anchor Guide`）与 5 帧历史无缝接续，并保留原视频音轨。
- **Browser Access / 浏览器访问**: Uses **Google Colab Native Proxy** (`google.colab.kernel.proxyPort`) directly in your signed-in browser session — no password setup, no external tunnel, and no external API exposed; expires when the runtime stops.
- **Execution Order / 执行顺序**: `1. Environment Check` → `2. Install Engine & Web Bundle` → `3. Download Model Weights` → `4. Launch Web Studio`

### 1. Environment Check · 环境检测
- **What this does**: Verifies CUDA GPU model, VRAM, BF16 support, system RAM, and free disk space under `/content`.
- **中文说明**：检测 GPU 型号（推荐 `A100 80GB` / `H100` / `RTX 6000`，亦兼容 `40GB A100` 运行 `480p`）、可用磁盘空间与 BF16 支持。
- **Run**: Execute cell and wait for `[OK]`.

In [ ]:
# @title 1. Environment Check {display-mode: "form"}
import os, shutil, subprocess, torch

MIN_VRAM_GIB = 35
REQUIREMENT = "40GB+ VRAM (A100 40GB/80GB, H100 80GB, RTX 6000 96GB)"

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Select Runtime -> Change runtime type -> A100 / H100 / RTX 6000 GPU.")

print(subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout)

gpu = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_gib = shutil.disk_usage("/content").free / 1024**3
bf16 = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu} | VRAM: {vram_gib:.1f} GiB | RAM: {ram_gib:.1f} GiB | Disk Free: {disk_gib:.1f} GiB | BF16: {bf16}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

if not any(k in gpu.upper() for k in ("A100", "H100", "6000")) or vram_gib < MIN_VRAM_GIB:
    raise RuntimeError(f"Detected {gpu} ({vram_gib:.1f} GiB), but this notebook requires {REQUIREMENT}. Please switch runtime.")
if not bf16:
    raise RuntimeError("Current GPU does not support BF16; please switch runtime.")
if vram_gib < 70:
    print("[Note] <70 GiB VRAM detected (40GB tier): use 480p resolution and start with 5s test clips first.")
print(f"[OK] GPU environment ready: {gpu} ({vram_gib:.1f} GiB)")

### 2. Install Inference Engine & App Bundle · 安装推理引擎与应用包
- **What this does**: Checks out pinned ComfyUI (`v0.43.2`), installs `av>=19.0.1` + Web Studio deps, verifies & extracts `app-h3-character-swap.tar.gz` (`v0.2.4`), and syncs the bundled `ComfyUI-H3-Swap-Mask` helper nodes.
- **中文说明**：安装固定版本内部无头 ComfyUI 引擎并拉取校验 Release 前端与后端应用包，同步内置 `ComfyUI-H3-Swap-Mask` 遮罩辅助节点。
- **Run**: Execute cell and wait for `[OK]`.

In [ ]:
# @title 2. Install Inference Engine & App Bundle {display-mode: "form"}
import hashlib, re, shutil, subprocess, sys, tarfile, tempfile, urllib.request
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
COMFY_REPO = "https://github.com/Comfy-Org/ComfyUI"
COMFY_VERSION = "v0.43.2"
COMFY_REVISION = "423cbd20511becb1f16122ab518e92e58bfa03d9"

APP_DIR = Path("/content/app")
RELEASE_REPO = "boltguo/colab"
RELEASE_TAG = "v0.2.4"
APP_ARCHIVE = "app-h3-character-swap.tar.gz"
APP_SHA256 = "d7e97c965739baf30459bccf330c4cf9fd7de0399fa6adbdedb08de878ade6a4"
BUNDLED_NODES_REL = "backend/features/h3_character_swap/comfy_nodes"

EXTRA_PACKAGES = ["av>=19.0.1"]
APP_SERVER_PACKAGES = [
    "fastapi>=0.115.0,<1.0.0",
    "uvicorn[standard]>=0.30.0,<1.0.0",
    "python-multipart>=0.0.9,<1.0.0",
    "pydantic>=2.7.0,<3.0.0",
    "huggingface_hub>=0.23.0",
    "hf_xet",
]

def run(args, cwd=None):
    subprocess.run(args, cwd=cwd, check=True)

def git(path, *args):
    return subprocess.check_output(["git", *args], cwd=path, text=True).strip()

def ensure_not_symlink(path, label):
    for p in [path, *path.parents]:
        if p == Path("/content"):
            break
        if p.is_symlink():
            raise RuntimeError(f"{label} path contains a symlink; start a fresh runtime: {p}")

def checkout_repo(path, repo_url, revision, label, allow_local_changes=False):
    ensure_not_symlink(path, label)
    fresh = not path.exists()
    if fresh:
        path.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--depth", "1", "--no-checkout", repo_url + ".git", str(path)])
    if not (path / ".git").is_dir():
        raise RuntimeError(f"{label} is not a git repository; please use a clean runtime: {path}")
    if git(path, "remote", "get-url", "origin").rstrip("/").removesuffix(".git") != repo_url:
        raise RuntimeError(f"Unexpected remote origin in {label}; please use a clean runtime.")
    if not fresh and not allow_local_changes and git(path, "status", "--porcelain"):
        raise RuntimeError(f"{label} has uncommitted local changes; please use a clean runtime.")
    if fresh or git(path, "rev-parse", "HEAD") != revision:
        run(["git", "fetch", "--depth", "1", "origin", revision], cwd=path)
        run(["git", "checkout", "--force", "--detach", revision], cwd=path)

def sync_bundled_custom_nodes():
    if not BUNDLED_NODES_REL:
        return
    src_root = APP_DIR / BUNDLED_NODES_REL
    if not src_root.is_dir():
        return
    custom_nodes_dir = COMFY_DIR / "custom_nodes"
    ensure_not_symlink(custom_nodes_dir, "ComfyUI custom_nodes")
    custom_nodes_dir.mkdir(parents=True, exist_ok=True)
    for node_dir in sorted(p for p in src_root.iterdir() if p.is_dir()):
        if node_dir.is_symlink() or any(child.is_symlink() for child in node_dir.rglob("*")):
            raise RuntimeError(f"Refusing symlink inside bundled custom nodes: {node_dir}")
        dest = custom_nodes_dir / node_dir.name
        if dest.is_symlink():
            raise RuntimeError(f"Destination custom node path is a symlink: {dest}")
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(node_dir, dest, symlinks=False)
        print(f"Synced bundled ComfyUI node pack: {node_dir.name}")

def install_release_app():
    ensure_not_symlink(APP_DIR, "App directory")
    stamp_file = APP_DIR / ".release_sha256"
    if (
        stamp_file.is_file()
        and not stamp_file.is_symlink()
        and stamp_file.read_text(encoding="utf-8").strip() == f"{RELEASE_TAG}:{APP_SHA256}"
        and (APP_DIR / "backend/run_server.py").is_file()
    ):
        print(f"App bundle already up to date: {APP_ARCHIVE} ({RELEASE_TAG})")
        return

    local_archive = Path(f"/content/{APP_ARCHIVE}")
    with tempfile.NamedTemporaryFile(suffix=".tar.gz", delete=False) as tmp:
        tmp_path = Path(tmp.name)
    try:
        if local_archive.is_file() and not local_archive.is_symlink():
            print(f"Using pre-staged app archive: {local_archive}")
            payload = local_archive.read_bytes()
        else:
            url = f"https://github.com/{RELEASE_REPO}/releases/download/{RELEASE_TAG}/{APP_ARCHIVE}"
            print(f"Downloading app bundle: {url}")
            req = urllib.request.Request(url, headers={"User-Agent": "colab-release-fetcher"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                payload = resp.read()
        actual_sha = hashlib.sha256(payload).hexdigest()
        if actual_sha != APP_SHA256:
            raise RuntimeError(
                f"App bundle SHA256 mismatch! Expected {APP_SHA256}, got {actual_sha}"
            )
        tmp_path.write_bytes(payload)

        staging = APP_DIR.with_name("app.staging")
        if staging.is_symlink():
            raise RuntimeError("Staging directory is a symlink; aborting.")
        if staging.exists():
            shutil.rmtree(staging)
        staging.mkdir(parents=True, exist_ok=True)
        staging_root = staging.resolve()

        with tarfile.open(tmp_path, "r:gz") as tf:
            for member in tf.getmembers():
                if member.issym() or member.islnk():
                    raise RuntimeError(f"Refusing symlink/hardlink archive entry: {member.name}")
                target = (staging / member.name).resolve()
                if target != staging_root and not str(target).startswith(str(staging_root) + "/"):
                    raise RuntimeError(f"Archive path escapes staging root: {member.name}")
            tf.extractall(staging)

        (staging / ".release_sha256").write_text(f"{RELEASE_TAG}:{APP_SHA256}\n", encoding="utf-8")
        if APP_DIR.exists():
            shutil.rmtree(APP_DIR)
        staging.replace(APP_DIR)
    finally:
        if tmp_path.exists():
            tmp_path.unlink()

checkout_repo(COMFY_DIR, COMFY_REPO, COMFY_REVISION, "ComfyUI")
REQUIREMENT_FILES = [COMFY_DIR / "requirements.txt"]

skip = {"torch", "torchvision", "torchaudio", "xformers"}
lines = []
for req_file in REQUIREMENT_FILES:
    for line in req_file.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        name = re.split(r"[<>=!~\[;\s]", s, maxsplit=1)[0].strip().lower()
        if name not in skip:
            lines.append(s)
requirements = Path("/content/comfy_requirements.txt")
requirements.write_text("\n".join(lines) + "\n", encoding="utf-8")
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements), *EXTRA_PACKAGES, *APP_SERVER_PACKAGES])

install_release_app()
sync_bundled_custom_nodes()

print(f"Engine: ComfyUI {COMFY_VERSION} ({COMFY_REVISION[:12]}) + H3-Swap-Mask | app: {RELEASE_TAG} ({APP_SHA256[:12]})")
print("[OK] Internal inference engine and Web Studio bundle installed.")

### 3. Download & Verify Model Weights · 下载并校验模型权重
- **What this does**: Downloads pinned weights `MiniMax-H3 Ref2VA INT8`, `FL2V Turbo 8-step`, `Character Swap LoRA V1.5`, `SAM 3.1 Multiplex`, `Qwen3-VL-32B-INT8`, `H3 Video VAE` & `H3 Audio VAE`（基于 Ref2VA + SAM 3.1 遮罩的人物与多人物替换） (53.8 GiB) into `/content/ComfyUI/models` using high-speed `hf_xet` transfers and exact byte-size checks.
- **中文说明**：使用 `hf_xet` 高速下载并精确校验 `MiniMax-H3 Ref2VA (INT8)`、`FL2V Turbo 8-step`、`Character Swap LoRA V1.5`、`SAM 3.1 Multiplex`、`Qwen3-VL-32B-INT8` 与双模态 VAE（共约 53.8 GiB）。
- **Run**: Execute once; existing verified files are reused automatically.

In [ ]:
# @title 3. Download & Verify Base Model Weights {display-mode: "form"}
import os, shutil, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
from huggingface_hub import hf_hub_download

COMFY_DIR = Path("/content/ComfyUI")
MODEL_DIR = COMFY_DIR / "models"
H3_SWAP_BASE = ("Comfy-Org/MiniMax-H3", "4b53a35191437de05370da548e59aeabe2a5a09a", "")
H3_SWAP_LORA = ("akatz-ai/MiniMax-H3-Character-Swap-LoRA", "96ea3407b230b6f7df71c58c70e31df96bd7e533", "loras")
SAM3 = ("Comfy-Org/sam3.1", "7bb8374780a725b4353ed31f3a9395c9742b5621", "")

MODEL_FILES = [
    (*H3_SWAP_BASE, "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors", 20970379616),
    (*H3_SWAP_BASE, "loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", 1956193000),
    (*H3_SWAP_LORA, "h3_character_swap_v1.5.safetensors", 155110792),
    (*SAM3, "checkpoints/sam3.1_multiplex_fp16.safetensors", 1745546848),
    (*H3_SWAP_BASE, "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors", 27141342152),
    (*H3_SWAP_BASE, "vae/minimax_h3_video_vae_fp16.safetensors", 5207808496),
    (*H3_SWAP_BASE, "vae/minimax_h3_audio_vae_fp32.safetensors", 605254808),
]

def local_directory(path):
    path = Path(path)
    for parent in [path, *path.parents]:
        if parent == COMFY_DIR.parent:
            break
        if parent.is_symlink():
            raise RuntimeError(f"Directory is a symlink; please start a fresh runtime: {parent}")
    path.mkdir(parents=True, exist_ok=True)

def model_ready(target, size):
    return target.is_file() and not target.is_symlink() and target.stat().st_size == size

def download_model(repo, revision, folder, file, size):
    base = MODEL_DIR / folder if folder else MODEL_DIR
    target = base / file
    local_directory(target.parent)
    started = time.time()
    result = Path(
        hf_hub_download(
            repo_id=repo,
            filename=file,
            revision=revision,
            local_dir=str(base),
            force_download=target.exists(),
            token=False,
        )
    )
    if result.resolve() != target.resolve() or not model_ready(target, size):
        raise RuntimeError(f"Incomplete or mismatched model download: {target}")
    print(f"Downloaded: {file} ({size / 1024**3:.2f} GiB in {time.time() - started:.0f}s)")

def download_models():
    pending = []
    for item in MODEL_FILES:
        base = MODEL_DIR / item[2] if item[2] else MODEL_DIR
        target = base / item[3]
        if target.is_symlink():
            raise RuntimeError(f"Refusing symlink model file: {target}")
        if model_ready(target, item[4]):
            print("Verified:", item[3])
        else:
            pending.append(item)
    if not pending:
        return
    missing = sum(item[4] for item in pending)
    free = shutil.disk_usage(MODEL_DIR).free
    if free < missing + 2 * 1024**3:
        raise RuntimeError(f"Insufficient disk space: need {missing / 1024**3:.1f} GiB more, free {free / 1024**3:.1f} GiB.")
    started = time.time()
    pool = ThreadPoolExecutor(max_workers=3)
    try:
        futures = [pool.submit(download_model, *item) for item in sorted(pending, key=lambda item: -item[4])]
        for future in as_completed(futures):
            future.result()
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
    elapsed = time.time() - started
    print(f"Base weights ready: {missing / 1024**3:.1f} GiB in {elapsed:.0f}s ({missing / 1024**2 / max(elapsed, 1):.0f} MB/s)")

for path in [MODEL_DIR, MODEL_DIR / "loras", COMFY_DIR / "input", COMFY_DIR / "output"]:
    local_directory(path)
download_models()
print("[OK] Base model weights verified.")

### 4. Launch Web Studio (Colab Native Proxy) · 启动 Web 工作台（Colab 原生代理）
- **What this does**: Starts internal ComfyUI on `127.0.0.1:8189` (whitelisting `ComfyUI-H3-Swap-Mask`), launches Web Studio on `127.0.0.1:8188`, and opens it through Google Colab's native browser proxy.
- **中文说明**：后台启动 ComfyUI 推理引擎与 Web 工作台，并通过 Colab 原生端口代理生成仅当前 Google 账号可打开的浏览器入口（无需密码、无 Cloudflare 隧道、不开放外部 API）。
- **Run**: Execute inside your browser Colab notebook and click the opened window link.

In [ ]:
# @title 4. Launch Web Studio {display-mode: "form"}
import json, os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path
from urllib.parse import urlsplit

FEATURE_NAME = "h3-character-swap"
BUNDLED_NODES_REL = "backend/features/h3_character_swap/comfy_nodes"
COMFY_DIR = Path("/content/ComfyUI")
COMFY_PORT = 8189
COMFY_BASE = f"http://127.0.0.1:{COMFY_PORT}"
COMFY_LOG = Path("/content/comfyui-worker.log")
COMFY_LAUNCHER = Path("/content/comfy_userdata_compat.py")
COMFY_LAUNCH_CONFIG = Path("/content/comfyui-worker.launch.json")
COMFY_PROCESS_PATTERN = r"comfy_userdata_compat\.py|ComfyUI/main\.py"
COMFY_LAUNCH_ARGS = [
    "--output-directory", str(COMFY_DIR / "output"),
    "--disable-all-custom-nodes",
    "--vram-headroom", "4",
    "--use-pytorch-cross-attention",
    "--whitelist-custom-nodes", "ComfyUI-H3-Swap-Mask",
    "--disable-partner-nodes",
]
REQUIRED_NODES = {"UNETLoader", "LoraLoaderModelOnly", "CLIPLoader", "VAELoader", "CheckpointLoaderSimple", "CLIPTextEncode", "LoadVideo", "GetVideoComponents", "LoadImage", "ImageFromBatch", "ImageScale", "VAEEncode", "SAM3_Detect", "SAM3_VideoTrack", "SAM3_TrackToMask", "H3SwapSelectMask", "H3SwapExpandMask", "H3SwapApplyEditMask", "MiniMaxH3SigmaShift", "MiniMaxH3ReferenceToVideo", "MiniMaxH3AddGuide", "RandomNoise", "BasicGuider", "KSamplerSelect", "BasicScheduler", "SamplerCustomAdvanced", "LTXVSeparateAVLatent", "VAEDecode", "CreateVideo", "SaveVideo"}

APP_DIR = Path("/content/app")
DATA_DIR = Path(f"/content/app_data/{FEATURE_NAME}")
PORT = 8188
BASE = f"http://127.0.0.1:{PORT}"
APP_LOG = Path(f"/content/{FEATURE_NAME}-web.log")
APP_LAUNCH_CONFIG = Path(f"/content/{FEATURE_NAME}.launch.json")
APP_PROCESS_PATTERN = r"backend/run_server\.py|backend\.run_server"

def read_text(path):
    return path.read_text(errors="replace") if path.exists() else ""

def tail(path, size=8000):
    return read_text(path)[-size:]

def rotate_log(path):
    if path.is_file() and path.stat().st_size:
        path.replace(path.with_name(path.name + ".1"))

def process_running(pattern):
    return subprocess.run(["pgrep", "-f", pattern], stdout=subprocess.DEVNULL).returncode == 0

def stop_processes(pattern, timeout):
    subprocess.run(["pkill", "-f", pattern], check=False)
    deadline = time.time() + timeout
    while process_running(pattern) and time.time() < deadline:
        time.sleep(2)
    if process_running(pattern):
        subprocess.run(["pkill", "-9", "-f", pattern], check=False)
        time.sleep(2)

def resolve_colab_proxy_url(port):
    try:
        from google.colab import output
    except ImportError:
        return f"http://127.0.0.1:{int(port)}", "local"
    try:
        proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({int(port)})")
    except Exception as exc:
        raise RuntimeError(
            "Failed to resolve Colab native proxy URL. Run Step 4 inside a browser Google Colab notebook session."
        ) from exc
    if not isinstance(proxy_url, str) or not proxy_url.strip().startswith("https://"):
        raise RuntimeError(
            f"Colab did not return a valid HTTPS proxy URL ({proxy_url!r}). Open the notebook in your browser and re-run Step 4."
        )
    return proxy_url.strip().rstrip("/"), "colab"

def open_colab_proxy(port, proxy_url, access_mode):
    if access_mode == "colab":
        from google.colab import output
        output.serve_kernel_port_as_window(int(port))
        print(f"[OK] Web Studio (Colab Native Proxy): {proxy_url}")
        print("Access is restricted to your signed-in Google account and expires when this runtime stops.")
    else:
        print(f"[OK] Web Studio (Local): {proxy_url}")

if not (COMFY_DIR / "main.py").is_file():
    raise RuntimeError("Internal engine /content/ComfyUI not found; please run Step 2 first.")
if APP_DIR.is_symlink() or not (APP_DIR / "backend/run_server.py").is_file():
    raise RuntimeError("App directory /content/app not found; please run Step 2 first.")

if FEATURE_NAME in ("minimax-h3", "seedvr2", "h3-character-swap"):
    missing_media_tools = [name for name in ("ffmpeg", "ffprobe") if not shutil.which(name)]
    if missing_media_tools:
        raise RuntimeError(f"Missing required media tools: {', '.join(missing_media_tools)}. Run !apt-get -qq update && apt-get -qq install -y ffmpeg first.")

if BUNDLED_NODES_REL:
    bundled_nodes_src = APP_DIR / BUNDLED_NODES_REL
    if bundled_nodes_src.is_dir():
        custom_nodes_dir = COMFY_DIR / "custom_nodes"
        custom_nodes_dir.mkdir(parents=True, exist_ok=True)
        for node_dir in sorted(p for p in bundled_nodes_src.iterdir() if p.is_dir()):
            if node_dir.is_symlink() or any(c.is_symlink() for c in node_dir.rglob("*")):
                raise RuntimeError(f"Refusing symlink inside bundled custom nodes: {node_dir}")
            dest = custom_nodes_dir / node_dir.name
            if dest.is_symlink():
                raise RuntimeError(f"Destination custom node path is a symlink: {dest}")
            if dest.exists():
                shutil.rmtree(dest)
            shutil.copytree(node_dir, dest, symlinks=False)

LAUNCHER_SOURCE = r'''
"""Start pinned ComfyUI with narrow workflow aliases and Cloudflare compatibility."""
import os
import runpy
import sys
from pathlib import Path
from aiohttp import web


def add_workflow_aliases(manager, routes):
    originals = [route for route in routes
                 if getattr(route, "path", None) == "/userdata/{file}"
                 and getattr(route, "method", None) in {"GET", "POST", "DELETE"}]
    if {route.method for route in originals} != {"GET", "POST", "DELETE"}:
        raise RuntimeError("ComfyUI userdata routes changed; workflow compatibility aliases not installed.")
    for route in originals:
        async def workflow_alias(request, handler=route.handler):
            file = request.match_info.get("file", "")
            user_root = manager.get_request_user_filepath(request, None, create_dir=False)
            candidate = manager.get_request_user_filepath(request, file, create_dir=False)
            if not user_root or not candidate:
                return web.Response(status=403)
            workflow_directory = os.path.join(user_root, "workflows")
            if os.path.islink(workflow_directory):
                return web.Response(status=403)
            workflow_root = os.path.realpath(workflow_directory)
            try:
                within = os.path.commonpath((workflow_root, os.path.realpath(candidate))) == workflow_root
            except ValueError:
                within = False
            if not within or os.path.realpath(candidate) == workflow_root:
                return web.Response(status=403)
            return await handler(request)
        routes.route(route.method, "/userdata/{file:workflows/[^/]+}")(workflow_alias)


def install_cf_middleware():
    import middleware.cache_middleware as cm
    orig_cache_control = cm.cache_control

    @web.middleware
    async def cf_safe_middleware(request, handler):
        response = await orig_cache_control(request, handler)
        if request.method == "GET" and request.path == "/" and getattr(response, "status", None) == 200:
            import server
            inst = getattr(server.PromptServer, "instance", None)
            web_root = getattr(inst, "web_root", None) if inst else None
            index_file = Path(web_root) / "index.html" if web_root else None
            if index_file and index_file.is_file():
                html = index_file.read_text(encoding="utf-8", errors="replace")
                html = html.replace("<script ", '<script data-cfasync="false" ')
                new_resp = web.Response(text=html, content_type="text/html")
                for k, v in response.headers.items():
                    if k.lower() not in {"content-length", "content-type", "etag", "last-modified"}:
                        new_resp.headers[k] = v
                new_resp.headers["Cache-Control"] = "no-store, no-transform, must-revalidate"
                response = new_resp
        if isinstance(response, web.StreamResponse) and not isinstance(response, web.WebSocketResponse) and not response.prepared:
            if getattr(response, "status", None) == 200 and (
                request.path.startswith("/assets/")
                or request.path.endswith((".woff2", ".woff", ".ttf"))
            ):
                response.headers["Cache-Control"] = "public, max-age=604800, immutable, no-transform"
            else:
                cc = response.headers.get("Cache-Control", "")
                if "no-transform" not in cc:
                    response.headers["Cache-Control"] = f"{cc}, no-transform" if cc else "no-transform"
            origin = request.headers.get("Origin")
            if origin and response.headers.get("Access-Control-Allow-Origin") == "*":
                response.headers["Access-Control-Allow-Origin"] = origin
                response.headers["Vary"] = "Origin"
        return response

    cm.cache_control = cf_safe_middleware


def main():
    comfy_root = Path("/content/ComfyUI")
    if comfy_root.is_symlink() or not (comfy_root / "main.py").is_file() or not (comfy_root / "comfy/options.py").is_file():
        raise RuntimeError("Missing or invalid ComfyUI directory.")
    os.chdir(comfy_root)
    sys.path.insert(0, str(comfy_root))
    sys.argv[0] = str(comfy_root / "main.py")
    import comfy.options
    comfy.options.enable_args_parsing()
    install_cf_middleware()
    from app.user_manager import UserManager
    original = UserManager.add_routes

    def compatible_routes(self, routes):
        original(self, routes)
        add_workflow_aliases(self, routes)

    UserManager.add_routes = compatible_routes
    runpy.run_path("main.py", run_name="__main__")


if __name__ == "__main__":
    main()
'''

def comfy_get_json(route, timeout=5):
    with urllib.request.urlopen(COMFY_BASE + route, timeout=timeout) as response:
        return json.load(response)

def comfy_ready(timeout=5):
    try:
        comfy_get_json("/system_stats", timeout)
        return True
    except Exception:
        return False

# 1. Start internal headless ComfyUI worker (127.0.0.1:8189, loopback only)
comfy_cfg_str = json.dumps({"launcher": LAUNCHER_SOURCE, "args": COMFY_LAUNCH_ARGS, "port": COMFY_PORT})
comfy_running = comfy_ready() or (process_running(COMFY_PROCESS_PATTERN) and comfy_ready(timeout=30))
if comfy_running and read_text(COMFY_LAUNCH_CONFIG) != comfy_cfg_str:
    print("Restarting internal ComfyUI worker with updated launch configuration...")
    stop_processes(COMFY_PROCESS_PATTERN, 60)
    comfy_running = False

if comfy_running:
    print("[OK] Internal ComfyUI worker already running (127.0.0.1:8189).")
else:
    stop_processes(COMFY_PROCESS_PATTERN, 60)
    COMFY_LAUNCHER.write_text(LAUNCHER_SOURCE, encoding="utf-8")
    rotate_log(COMFY_LOG)
    with COMFY_LOG.open("w") as log:
        comfy_proc = subprocess.Popen(
            [sys.executable, str(COMFY_LAUNCHER), "--listen", "127.0.0.1", "--port", str(COMFY_PORT), *COMFY_LAUNCH_ARGS],
            cwd=COMFY_DIR,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    COMFY_LAUNCH_CONFIG.write_text(comfy_cfg_str, encoding="utf-8")
    print("Starting internal GPU inference engine...")
    deadline = time.time() + 600
    while time.time() < deadline:
        if comfy_proc.poll() is not None:
            print(tail(COMFY_LOG))
            raise RuntimeError(f"Internal engine failed to start (exit code {comfy_proc.returncode}); see logs above.")
        if comfy_ready():
            print("[OK] Internal GPU inference engine ready (127.0.0.1:8189).")
            break
        time.sleep(2)
    else:
        print(tail(COMFY_LOG))
        raise TimeoutError("Internal engine did not become ready within 10 minutes; check logs.")

missing = sorted(REQUIRED_NODES - set(comfy_get_json("/object_info", timeout=60)))
if missing:
    print(tail(COMFY_LOG))
    raise RuntimeError(f"Missing required ComfyUI nodes: {', '.join(missing)}. Re-run Step 2.")

# 2. Start Web Studio server (127.0.0.1:8188, loopback only)
def app_ready(timeout=5):
    try:
        with urllib.request.urlopen(BASE + "/api/v1/health", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

proxy_url, access_mode = resolve_colab_proxy_url(PORT)
proxy_origin = f"{urlsplit(proxy_url).scheme}://{urlsplit(proxy_url).netloc}" if access_mode == "colab" else ""

env_overrides = {
    "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
    "COLAB_FEATURE": FEATURE_NAME,
    "COLAB_HOST": "127.0.0.1",
    "COLAB_PORT": str(PORT),
    "COLAB_DATA_DIR": str(DATA_DIR),
    "COLAB_DIST_DIR": str(APP_DIR / "dist" / FEATURE_NAME),
    "COLAB_PROXY_ORIGIN": proxy_origin,
    "COMFY_BASE_URL": COMFY_BASE,
    "COMFY_DIR": str(COMFY_DIR),
}

app_cmd = [
    sys.executable,
    str(APP_DIR / "backend/run_server.py"),
    "--feature",
    FEATURE_NAME,
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
    "--dist-dir",
    str(APP_DIR / "dist" / FEATURE_NAME),
]

app_fingerprint = json.dumps({
    "cmd": app_cmd,
    "release": read_text(APP_DIR / ".release_sha256").strip(),
    "env": env_overrides,
}, sort_keys=True)
app_is_up = app_ready() or (process_running(APP_PROCESS_PATTERN) and app_ready(timeout=15))
if app_is_up and read_text(APP_LAUNCH_CONFIG) != app_fingerprint:
    print("Web Studio configuration changed; restarting...")
    stop_processes(APP_PROCESS_PATTERN, 30)
    app_is_up = False

if app_is_up:
    print("[OK] Web Studio server already running (127.0.0.1:8188).")
else:
    stop_processes(APP_PROCESS_PATTERN, 30)
    rotate_log(APP_LOG)
    child_env = {**os.environ, **env_overrides}
    with APP_LOG.open("w") as log:
        app_proc = subprocess.Popen(
            app_cmd,
            cwd=APP_DIR,
            env=child_env,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    APP_LAUNCH_CONFIG.write_text(app_fingerprint, encoding="utf-8")
    print("Starting Web Studio...")
    deadline = time.time() + 120
    while time.time() < deadline:
        if app_proc.poll() is not None:
            print(tail(APP_LOG))
            raise RuntimeError(f"Web Studio failed to start (exit code {app_proc.returncode}); see logs above.")
        if app_ready():
            print("[OK] Web Studio started.")
            break
        time.sleep(1.5)
    else:
        print(tail(APP_LOG))
        raise TimeoutError("Web Studio did not become ready within 2 minutes; check logs.")

# 3. Expose via Google Colab native proxy & persist live URL metadata
if not DATA_DIR.is_symlink():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    proxy_info_path = DATA_DIR / "proxy_info.json"
    if not proxy_info_path.is_symlink():
        proxy_info_path.write_text(
            json.dumps({"public_url": proxy_url, "access_mode": access_mode}, ensure_ascii=False),
            encoding="utf-8",
        )
open_colab_proxy(PORT, proxy_url, access_mode)